# Multi-Agent System — Async / Parallel Workers

Same architecture as `multi_agent.ipynb` but worker agents (Planner, Executor, Critic)
are dispatched **concurrently** using `asyncio` when the Orchestrator fires multiple
tool calls in a single response.

**Key changes from `multi_agent.ipynb`:**
- `anthropic.AsyncAnthropic` client
- `_worker_call` / `dispatch` / `run` are all `async`
- `asyncio.gather()` runs all tool calls in a batch in parallel

**Cost impact:** Identical to the sync version — same token counts, same number of API calls.
Benefit is reduced wall-clock time when the Orchestrator batches multiple tool calls in one response.

## Imports & Config

In [1]:
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import asyncio
import json
import os
import anthropic

ORCHESTRATOR_MODEL      = "claude-sonnet-4-6"
WORKER_MODEL            = "claude-haiku-4-5-20251001"

MAX_ORCHESTRATOR_TOKENS = 2048
MAX_WORKER_TOKENS       = 1024
MAX_ITERATIONS          = 10

# AsyncAnthropic instead of Anthropic
client = anthropic.AsyncAnthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

## Cost Tracker

In [4]:
COST_PER_M = {
    ORCHESTRATOR_MODEL: {"input": 3.00,  "output": 15.00},
    WORKER_MODEL:       {"input": 0.80,  "output":  4.00},
}

total_cost_usd = 0.0

def log_usage(model: str, usage: anthropic.types.Usage, label: str) -> None:
    global total_cost_usd
    rates = COST_PER_M.get(model, {"input": 3.0, "output": 15.0})

    cache_read    = getattr(usage, "cache_read_input_tokens", 0) or 0
    cache_written = getattr(usage, "cache_creation_input_tokens", 0) or 0
    billed_input  = usage.input_tokens - cache_read

    cost = (
        (billed_input / 1_000_000) * rates["input"]
        + (cache_read / 1_000_000) * rates["input"] * 0.10
        + (usage.output_tokens / 1_000_000) * rates["output"]
    )
    total_cost_usd += cost

    print(
        f"  📊 {label} | in={usage.input_tokens}"
        + (f" (cache_hit={cache_read}, cache_write={cache_written})" if cache_read or cache_written else "")
        + f" | out={usage.output_tokens} | ${cost:.5f}"
    )

## System Prompts

In [5]:
ORCHESTRATOR_SYSTEM = """\
You are an Orchestrator that coordinates three specialized AI agents to complete tasks.

Agents available (call via tools):
  call_planner   — breaks a goal into ordered subtasks
  call_executor  — executes one subtask and returns concrete output
  call_critic    — reviews output against criteria, returns score + feedback

Standard workflow:
  1. call_planner  → get subtask list
  2. call_executor → for each subtask (pass context from completed steps)
  3. call_critic   → on the final combined output
  4. If critic score < 7, re-execute failing subtasks with its feedback
  5. Return a clear final answer to the user

Be efficient. Do not call agents unnecessarily.\
"""

PLANNER_SYSTEM = """\
You are a Planner. Decompose the given goal into clear, ordered, actionable subtasks.

Rules:
  - Output ONLY a numbered list. No preamble, no explanation.
  - Each subtask: specific, independently actionable, 1-2 sentences.
  - 3-6 subtasks is ideal. Never more than 8.\
"""

EXECUTOR_SYSTEM = """\
You are an Executor. You receive one subtask and must complete it thoroughly.

Rules:
  - Think step by step internally, then output only the concrete result.
  - Be specific — no vague summaries.
  - If context from prior steps is provided, use it.\
"""

CRITIC_SYSTEM = """\
You are a Critic. Evaluate the provided output against the stated criteria.

Return ONLY a JSON object in this exact shape — no other text:
{
  "score": <1-10>,
  "passed": <true|false>,
  "issues": ["<issue>", ...],
  "suggestions": ["<suggestion>", ...]
}\
"""

## Async Worker Agent Calls

All workers are now `async` — they can be awaited concurrently via `asyncio.gather()`.

In [6]:
async def _worker_call(label: str, system: str, user_message: str) -> str:
    """Single async Haiku call with prompt caching on the system prompt."""
    response = await client.messages.create(
        model=WORKER_MODEL,
        max_tokens=MAX_WORKER_TOKENS,
        system=[{
            "type": "text",
            "text": system,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": user_message}],
    )
    log_usage(WORKER_MODEL, response.usage, label)
    return response.content[0].text


async def call_planner(goal: str) -> str:
    return await _worker_call("Planner", PLANNER_SYSTEM, f"Goal: {goal}")


async def call_executor(subtask: str, context: str = "") -> str:
    msg = f"Subtask: {subtask}"
    if context:
        msg += f"\n\nContext from previous steps:\n{context}"
    return await _worker_call("Executor", EXECUTOR_SYSTEM, msg)


async def call_critic(output: str, criteria: str) -> str:
    return await _worker_call(
        "Critic",
        CRITIC_SYSTEM,
        f"Output to review:\n{output}\n\nCriteria:\n{criteria}",
    )

## Tool Definitions

In [7]:
AGENT_TOOLS = [
    {
        "name": "call_planner",
        "description": "Decompose a high-level goal into ordered, actionable subtasks.",
        "input_schema": {
            "type": "object",
            "properties": {
                "goal": {"type": "string", "description": "The goal to decompose."},
            },
            "required": ["goal"],
        },
    },
    {
        "name": "call_executor",
        "description": (
            "Execute a single subtask and return concrete output. "
            "Pass context from already-completed subtasks to help this one."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "subtask": {"type": "string", "description": "The subtask to execute."},
                "context": {
                    "type": "string",
                    "description": "Relevant output from previous completed subtasks.",
                },
            },
            "required": ["subtask"],
        },
    },
    {
        "name": "call_critic",
        "description": (
            "Review output against criteria. Returns JSON with score (1-10), "
            "passed flag, issues, and suggestions."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "output":   {"type": "string", "description": "The output to review."},
                "criteria": {"type": "string", "description": "What to evaluate against."},
            },
            "required": ["output", "criteria"],
        },
    },
]

## Async Tool Dispatch

In [8]:
async def dispatch(tool_name: str, tool_input: dict) -> str:
    match tool_name:
        case "call_planner":
            return await call_planner(tool_input["goal"])
        case "call_executor":
            return await call_executor(tool_input["subtask"], tool_input.get("context", ""))
        case "call_critic":
            return await call_critic(tool_input["output"], tool_input["criteria"])
        case _:
            return json.dumps({"error": f"Unknown agent: {tool_name}"})

## Orchestrator Loop — Parallel Tool Dispatch

The key change: `asyncio.gather()` fires all tool calls in a batch simultaneously
instead of one at a time.

In [9]:
async def run(goal: str) -> str:
    """Run the multi-agent loop for a given goal. Returns the final answer."""
    global total_cost_usd
    total_cost_usd = 0.0

    print(f"\n{'═'*60}")
    print(f"Goal: {goal}")
    print(f"{'═'*60}")

    messages = [{"role": "user", "content": goal}]

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n[Iteration {iteration}]")

        response = await client.messages.create(
            model=ORCHESTRATOR_MODEL,
            max_tokens=MAX_ORCHESTRATOR_TOKENS,
            system=ORCHESTRATOR_SYSTEM,
            tools=AGENT_TOOLS,
            messages=messages,
        )
        log_usage(ORCHESTRATOR_MODEL, response.usage, "Orchestrator")

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\n{'─'*60}")
                    print(f"Total cost: ${total_cost_usd:.5f}")
                    print(f"{'─'*60}")
                    return block.text
            return "No response generated."

        if response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})

            # Collect all tool_use blocks in this response
            tool_use_blocks = [b for b in response.content if b.type == "tool_use"]

            for b in tool_use_blocks:
                print(f"  → {b.name}({json.dumps(b.input)[:80]}...)")

            # Dispatch ALL tool calls concurrently
            results = await asyncio.gather(
                *[dispatch(b.name, b.input) for b in tool_use_blocks]
            )

            # Build tool_result turn
            tool_results = []
            for block, result in zip(tool_use_blocks, results):
                print(f"  ← {block.name} returned {len(result)} chars")
                tool_results.append({
                    "type":        "tool_result",
                    "tool_use_id": block.id,
                    "content":     result,
                })

            messages.append({"role": "user", "content": tool_results})

        else:
            print(f"  Unexpected stop_reason: {response.stop_reason}")
            break

    return f"Reached max iterations ({MAX_ITERATIONS}). Check logs above for partial output."

## Run

Set your goal below and execute the cell.

> **Note:** Jupyter already runs an event loop, so `asyncio.run()` won't work here.
> Use `await run(goal)` directly instead.

In [10]:
goal = "Write a concise market analysis for electric vehicles in Canada"

result = await run(goal)

print(f"\n{'═'*60}")
print("FINAL RESULT")
print(f"{'═'*60}")
print(result)


════════════════════════════════════════════════════════════
Goal: Write a concise market analysis for electric vehicles in Canada
════════════════════════════════════════════════════════════

[Iteration 1]
  📊 Orchestrator | in=961 | out=65 | $0.00386
  → call_planner({"goal": "Write a concise market analysis for electric vehicles in Canada"}...)
  📊 Planner | in=103 | out=149 | $0.00068
  ← call_planner returned 716 chars

[Iteration 2]
  📊 Orchestrator | in=1182 | out=463 | $0.01049
  → call_executor({"subtask": "Research current EV market size, growth rate, and sales trends in C...)
  → call_executor({"subtask": "Identify key market drivers for EVs in Canada such as government in...)
  → call_executor({"subtask": "Analyze major EV manufacturers and models dominating the Canadian m...)
  → call_executor({"subtask": "Examine the competitive landscape and barriers to entry in Canada's...)
  → call_executor({"subtask": "Assess the regulatory environment for EVs in Canada, including fe